In [112]:
import os
import tarfile
import csv
import numpy as np
import pandas as pd
import gym
import torch
import warnings
warnings.filterwarnings("ignore")
import os
import tarfile
import csv
import numpy as np
import pandas as pd
import gym
import torch
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import warnings

warnings.filterwarnings("ignore")

DATA_DIR = os.path.join(os.getcwd(), "experiment_data", "training_data")

ACTIONS = [
    78.0, 83.0, 89.0, 95.0, 101.0, 107.0, 112.0, 118.0,
    124.0, 130.0, 136.0, 141.0, 147.0, 153.0, 159.0, 165.0
]

ACTION_TO_IDX = {a: i for i, a in enumerate(ACTIONS)}


In [113]:
def extract_tar_if_needed(app_dir, tar_name):
    extract_dir = os.path.join(app_dir, tar_name[:-4])
    if not os.path.exists(extract_dir):
        with tarfile.open(os.path.join(app_dir, tar_name), 'r') as tar:
            tar.extractall(extract_dir)
    return extract_dir


In [114]:
def generate_PCAP(df):
    df = df.copy()
    if 'wall_time' in df.columns:
        df.rename(columns={'wall_time': 'timestamp'}, inplace=True)
    df['elapsed_time'] = df['timestamp'] - df['timestamp'].iloc[0]
    return df[['timestamp', 'step_id', 'value']]


In [115]:
def compute_measured_power(df):
    df = df.copy()
    if 'wall_time' in df.columns:
        df.rename(columns={'wall_time': 'timestamp'}, inplace=True)
    return df[['timestamp', 'step_id', 'value']]


In [116]:
def median_by_step(df, value_col='value'):
    if df.empty:
        return {}
    return df.groupby('step_id')[value_col].median().to_dict()


In [117]:
def normalize_papi_scope(df):
    """
    Extracts the PAPI metric name from the full NRM scope string.

    Example:
      nrm.extra.perf.PAPI_TOT_INS.22869 -> PAPI_TOT_INS
    """
    df = df.copy()
    df['metric'] = df['scope'].str.extract(r'(PAPI_[A-Z_]+)')
    return df


In [118]:
def collect_derived_papi(papi_df):
    """
    Computes per-step derived PAPI metrics using stable samples only.
    """

    papi_df = normalize_papi_scope(papi_df)
    papi_df = papi_df[papi_df['stable'] == 1].copy()

    out = {}

    def step_delta(metric):
        df = papi_df[papi_df['metric'] == metric]

        if df.empty:
            return {}

        # Take last value per step
        last_vals = (
            df.sort_values('wall_time')
              .groupby('step_id')['value']
              .last()
        )

        # Delta between consecutive steps
        delta = last_vals.diff().fillna(0)

        return delta.to_dict()

    tot_ins = step_delta('PAPI_TOT_INS')
    tot_cyc = step_delta('PAPI_TOT_CYC')
    l3_tcm  = step_delta('PAPI_L3_TCM')
    l3_tca  = step_delta('PAPI_L3_TCA')
    stl     = step_delta('PAPI_RES_STL')

    derived = {}

    for step in set(tot_ins) | set(tot_cyc):
        derived.setdefault(step, {})['TOT_INS_PER_CYC'] = (
            tot_ins.get(step, 0) / (tot_cyc.get(step, 0) + 1e-9)
        )

    for step in set(l3_tcm) | set(l3_tca):
        derived.setdefault(step, {})['L3_TCM_PER_TCA'] = (
            l3_tcm.get(step, 0) / (l3_tca.get(step, 0) + 1e-9)
        )

    for step in set(stl) | set(tot_cyc):
        derived.setdefault(step, {})['TOT_STL_PER_CYC'] = (
            stl.get(step, 0) / (tot_cyc.get(step, 0) + 1e-9)
        )

    # Split into per-metric dicts (matching your downstream code)
    return {
        'TOT_INS_PER_CYC': {k: v.get('TOT_INS_PER_CYC', 0) for k, v in derived.items()},
        'L3_TCM_PER_TCA':  {k: v.get('L3_TCM_PER_TCA', 0) for k, v in derived.items()},
        'TOT_STL_PER_CYC': {k: v.get('TOT_STL_PER_CYC', 0) for k, v in derived.items()},
    }


In [119]:
training_data = {}

for app in os.listdir(DATA_DIR):
    app_dir = os.path.join(DATA_DIR, app)
    if not os.path.isdir(app_dir):
        continue

    training_data[app] = []

    for f in os.listdir(app_dir):
        if not f.endswith(".tar"):
            continue

        d = extract_tar_if_needed(app_dir, f)

        progress = pd.read_csv(f"{d}/progress.csv")
        power    = pd.read_csv(f"{d}/measured_power.csv")
        papi     = pd.read_csv(f"{d}/papi.csv")
        pcap     = pd.read_csv(f"{d}/PCAP_file.csv")

        # keep stable only
        if 'stable' in progress.columns:
            progress = progress[progress['stable'] == 1]
        if 'stable' in power.columns:
            power = power[power['stable'] == 1]
        if 'stable' in papi.columns:
            papi = papi[papi['stable'] == 1]

        pcap  = generate_PCAP(pcap)
        power = compute_measured_power(power)

        prog_median  = median_by_step(progress)
        power_median = median_by_step(power)
        papi_feats   = collect_derived_papi(papi)

        steps = sorted(pcap['step_id'].unique())

        for i in range(len(steps) - 1):
            s, ns = steps[i], steps[i+1]

            row = pcap[pcap['step_id'] == s]
            if row.empty:
                continue

            action = row['value'].iloc[0]

            state = np.array([
                prog_median.get(s, 0.0),
                power_median.get(s, 0.0),
                papi_feats['TOT_INS_PER_CYC'].get(s, 0.0),
                papi_feats['L3_TCM_PER_TCA'].get(s, 0.0),
                papi_feats['TOT_STL_PER_CYC'].get(s, 0.0),
            ])

            next_state = np.array([
                prog_median.get(ns, 0.0),
                power_median.get(ns, 0.0),
                papi_feats['TOT_INS_PER_CYC'].get(ns, 0.0),
                papi_feats['L3_TCM_PER_TCA'].get(ns, 0.0),
                papi_feats['TOT_STL_PER_CYC'].get(ns, 0.0),
            ])

            reward = (state[0] ** 3) / (state[1] + 1e-6)

            training_data[app].append((app, state, action, reward, next_state))


In [120]:
dataset = []
for app in training_data:
    dataset.extend(training_data[app])

print("Total transitions:", len(dataset))

out_csv = os.path.join(DATA_DIR, "training_dataset.csv")

with open(out_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        'App',
        'Progress', 'Power', 'TOT_INS_PER_CYC', 'L3_TCM_PER_TCA', 'TOT_STL_PER_CYC',
        'Action', 'Reward',
        'Next_Progress', 'Next_Power', 'Next_TOT_INS_PER_CYC',
        'Next_L3_TCM_PER_TCA', 'Next_TOT_STL_PER_CYC'
    ])

    for app, s, a, r, ns in dataset:
        writer.writerow([app, *s, a, r, *ns])


Total transitions: 45


In [121]:
loaded = pd.read_csv(out_csv)

training_dataset_loaded = [
    (
        tuple(row[1:6]),   # state
        row[6],            # action
        row[7],            # reward
        tuple(row[8:13])   # next_state
    )
    for _, row in loaded.iterrows()
]

print("Loaded transitions:", len(training_dataset_loaded))


Loaded transitions: 45


In [122]:
class OfflineEnv:
    def __init__(self, actions):
        self.actions = actions
        self.num_actions = len(actions)
        self.action_space = gym.spaces.Discrete(self.num_actions)

env = OfflineEnv(ACTIONS)


In [123]:
class FCNetwork(torch.nn.Module):
    def __init__(self, env, layers=[10,10]):
        super().__init__()
        dims = [5] + layers + [env.num_actions]
        net = []
        for i in range(len(dims)-1):
            net.append(torch.nn.Linear(dims[i], dims[i+1]))
            if i < len(dims)-2:
                net.append(torch.nn.ReLU())
        self.net = torch.nn.Sequential(*net)

    def forward(self, x):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.float32)
        return self.net(x)


In [124]:
def get_tensors(dataset, idxs):
    s, a, ns, r = [], [], [], []
    for i in idxs:
        s.append(dataset[i][0])
        a.append(dataset[i][1])
        r.append(dataset[i][2])
        ns.append(dataset[i][3])
    return np.array(s), np.array(a), np.array(ns), np.array(r)


In [125]:
def q_backup_sparse_sampled(network, ns, r, discount=0.99):
    with torch.no_grad():
        q_ns = network(ns)
    return r + discount * np.max(q_ns.numpy(), axis=1)


In [126]:
def project_qvalues_cql_sampled(s, a, target, network, optimizer, cql_alpha=0.0):
    s = torch.tensor(s, dtype=torch.float32)
    target = torch.tensor(target, dtype=torch.float32)

    a_idx = torch.tensor([ACTION_TO_IDX[x] for x in a], dtype=torch.int64)

    q = network(s)
    q_sa = q.gather(1, a_idx.view(-1,1)).squeeze()

    bellman = torch.mean((q_sa - target)**2)
    conservative = torch.mean(torch.logsumexp(q, dim=1) - q_sa)

    loss = bellman + cql_alpha * conservative

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()


In [127]:
def conservative_q_iteration(
    env, network, dataset,
    num_itrs=500, batch_size=128,
    lr=1e-5, cql_alpha=0.0, discount=0.9
):
    optimizer = torch.optim.RMSprop(network.parameters(), lr=lr)
    losses = []

    for it in range(num_itrs):
        idx = np.random.choice(len(dataset), batch_size)
        s, a, ns, r = get_tensors(dataset, idx)

        target = q_backup_sparse_sampled(network, ns, r, discount)
        loss = project_qvalues_cql_sampled(s, a, target, network, optimizer, cql_alpha)

        losses.append(loss)
        if it % 10 == 0:
            print(f"Iter {it:4d} | Loss {loss:.4f}")

    return losses


In [128]:
network = FCNetwork(env)
losses = conservative_q_iteration(
    env,
    network,
    training_dataset_loaded,
    num_itrs=500,
    cql_alpha=0.0,
    lr=1e-5
)


Iter    0 | Loss 69.7864
Iter   10 | Loss 70.2231
Iter   20 | Loss 70.1134
Iter   30 | Loss 53.4390


Iter   40 | Loss 59.6645
Iter   50 | Loss 51.8651
Iter   60 | Loss 53.2763
Iter   70 | Loss 60.8919
Iter   80 | Loss 56.8612
Iter   90 | Loss 58.5484
Iter  100 | Loss 53.0835
Iter  110 | Loss 66.9952
Iter  120 | Loss 63.0093
Iter  130 | Loss 56.3247
Iter  140 | Loss 66.7293
Iter  150 | Loss 63.7637
Iter  160 | Loss 58.9834
Iter  170 | Loss 47.6653
Iter  180 | Loss 50.5704
Iter  190 | Loss 49.6987
Iter  200 | Loss 71.8969
Iter  210 | Loss 49.3600
Iter  220 | Loss 48.8849
Iter  230 | Loss 52.5520
Iter  240 | Loss 63.0587
Iter  250 | Loss 51.7464
Iter  260 | Loss 50.6202
Iter  270 | Loss 49.8024
Iter  280 | Loss 53.6138
Iter  290 | Loss 52.9593
Iter  300 | Loss 50.9341
Iter  310 | Loss 36.4293
Iter  320 | Loss 51.8283
Iter  330 | Loss 54.5976
Iter  340 | Loss 45.9666
Iter  350 | Loss 52.0150
Iter  360 | Loss 62.5771
Iter  370 | Loss 54.0739
Iter  380 | Loss 47.0762
Iter  390 | Loss 56.2737
Iter  400 | Loss 50.1921
Iter  410 | Loss 53.0731
Iter  420 | Loss 51.3463
Iter  430 | Loss 45.1242
